<a href="https://colab.research.google.com/github/Namrata-1998-2053/kpi-detection-faiss/blob/main/NLP_based_KPI_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np

pd.set_option("display.max_colwidth", 200)

In [3]:
data = [
    # Positive examples
    {
        "session_id": "S001",
        "conversation": "Customer: I have been waiting for my claim for two weeks. Agent: I understand your frustration and I will check the claim status for you.",
        "kpi_met": 1
    },
    {
        "session_id": "S002",
        "conversation": "Customer: I am really worried about my delayed claim. Agent: I can understand why this situation is concerning. Let me look into it for you.",
        "kpi_met": 1
    },
    {
        "session_id": "S003",
        "conversation": "Customer: Nobody has updated me about my insurance claim. Agent: I completely understand your concern. I will check what is happening.",
        "kpi_met": 1
    },
    {
        "session_id": "S004",
        "conversation": "Customer: This delay is extremely frustrating. Agent: I can see why you would be frustrated. Let me help you with the issue.",
        "kpi_met": 1
    },
    {
        "session_id": "S005",
        "conversation": "Customer: I have called several times and still have no update. Agent: I understand how frustrating that must be. I will investigate this for you.",
        "kpi_met": 1
    },

    # Negative examples
    {
        "session_id": "S006",
        "conversation": "Customer: I have been waiting for my claim for two weeks. Agent: Your claim is still being processed.",
        "kpi_met": 0
    },
    {
        "session_id": "S007",
        "conversation": "Customer: I am worried about my delayed claim. Agent: Please provide your policy number so I can check the claim.",
        "kpi_met": 0
    },
    {
        "session_id": "S008",
        "conversation": "Customer: Nobody has updated me about my insurance claim. Agent: Your claim was submitted on Monday and is currently under review.",
        "kpi_met": 0
    },
    {
        "session_id": "S009",
        "conversation": "Customer: This delay is extremely frustrating. Agent: The standard processing time is ten business days.",
        "kpi_met": 0
    },
    {
        "session_id": "S010",
        "conversation": "Customer: I have called several times and still have no update. Agent: I will check the current status of your claim.",
        "kpi_met": 0
    }
]

df = pd.DataFrame(data)

df

,session_id,conversation,kpi_met
0,S001,Customer: I have been waiting for my claim for two weeks. Agent: I understand your frustration and I will check the claim status for you.,1
1,S002,Customer: I am really worried about my delayed claim. Agent: I can understand why this situation is concerning. Let me look into it for you.,1
2,S003,Customer: Nobody has updated me about my insurance claim. Agent: I completely understand your concern. I will check what is happening.,1
3,S004,Customer: This delay is extremely frustrating. Agent: I can see why you would be frustrated. Let me help you with the issue.,1
4,S005,Customer: I have called several times and still have no update. Agent: I understand how frustrating that must be. I will investigate this for you.,1
5,S006,Customer: I have been waiting for my claim for two weeks. Agent: Your claim is still being processed.,0
6,S007,Customer: I am worried about my delayed claim. Agent: Please provide your policy number so I can check the claim.,0
7,S008,Customer: Nobody has updated me about my insurance claim. Agent: Your claim was submitted on Monday and is currently under review.,0
8,S009,Customer: This delay is extremely frustrating. Agent: The standard processing time is ten business days.,0
9,S010,Customer: I have called several times and still have no update. Agent: I will check the current status of your claim.,0


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   session_id    10 non-null     object
 1   conversation  10 non-null     object
 2   kpi_met       10 non-null     int64 
dtypes: int64(1), object(2)
memory usage: 372.0+ bytes


In [5]:
df["kpi_met"].value_counts()

,count
kpi_met,
1,5
0,5


In [6]:
df["kpi_met"].value_counts(normalize=True)

,proportion
kpi_met,
1,0.5
0,0.5


In [7]:
import re

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r"customer:", "", text)
    text = re.sub(r"agent:", "", text)
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_conversation"] = df["conversation"].apply(preprocess_text)

df[["conversation", "clean_conversation"]]

,conversation,clean_conversation
0,Customer: I have been waiting for my claim for two weeks. Agent: I understand your frustration and I will check the claim status for you.,i have been waiting for my claim for two weeks i understand your frustration and i will check the claim status for you
1,Customer: I am really worried about my delayed claim. Agent: I can understand why this situation is concerning. Let me look into it for you.,i am really worried about my delayed claim i can understand why this situation is concerning let me look into it for you
2,Customer: Nobody has updated me about my insurance claim. Agent: I completely understand your concern. I will check what is happening.,nobody has updated me about my insurance claim i completely understand your concern i will check what is happening
3,Customer: This delay is extremely frustrating. Agent: I can see why you would be frustrated. Let me help you with the issue.,this delay is extremely frustrating i can see why you would be frustrated let me help you with the issue
4,Customer: I have called several times and still have no update. Agent: I understand how frustrating that must be. I will investigate this for you.,i have called several times and still have no update i understand how frustrating that must be i will investigate this for you
5,Customer: I have been waiting for my claim for two weeks. Agent: Your claim is still being processed.,i have been waiting for my claim for two weeks your claim is still being processed
6,Customer: I am worried about my delayed claim. Agent: Please provide your policy number so I can check the claim.,i am worried about my delayed claim please provide your policy number so i can check the claim
7,Customer: Nobody has updated me about my insurance claim. Agent: Your claim was submitted on Monday and is currently under review.,nobody has updated me about my insurance claim your claim was submitted on monday and is currently under review
8,Customer: This delay is extremely frustrating. Agent: The standard processing time is ten business days.,this delay is extremely frustrating the standard processing time is ten business days
9,Customer: I have called several times and still have no update. Agent: I will check the current status of your claim.,i have called several times and still have no update i will check the current status of your claim


In [8]:
!pip install -q sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 56.1 MB/s eta 0:00:00


In [9]:
from sentence_transformers import SentenceTransformer

In [10]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [11]:
embeddings = embedding_model.encode(
    df["clean_conversation"].tolist(),
    convert_to_numpy=True
)

print("Shape:", embeddings.shape)

Shape: (10, 384)


In [12]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(
    [embeddings[0]],
    [embeddings[1]]
)

print(similarity[0][0])

0.7079507


In [13]:
similarity = cosine_similarity(
    [embeddings[0]],
    [embeddings[5]]
)

print(similarity[0][0]
      )

0.8636636


In [14]:
baseline_df = df[df["kpi_met"] == 1].copy()

baseline_df

,session_id,conversation,kpi_met,clean_conversation
0,S001,Customer: I have been waiting for my claim for two weeks. Agent: I understand your frustration and I will check the claim status for you.,1,i have been waiting for my claim for two weeks i understand your frustration and i will check the claim status for you
1,S002,Customer: I am really worried about my delayed claim. Agent: I can understand why this situation is concerning. Let me look into it for you.,1,i am really worried about my delayed claim i can understand why this situation is concerning let me look into it for you
2,S003,Customer: Nobody has updated me about my insurance claim. Agent: I completely understand your concern. I will check what is happening.,1,nobody has updated me about my insurance claim i completely understand your concern i will check what is happening
3,S004,Customer: This delay is extremely frustrating. Agent: I can see why you would be frustrated. Let me help you with the issue.,1,this delay is extremely frustrating i can see why you would be frustrated let me help you with the issue
4,S005,Customer: I have called several times and still have no update. Agent: I understand how frustrating that must be. I will investigate this for you.,1,i have called several times and still have no update i understand how frustrating that must be i will investigate this for you


In [15]:
baseline_embeddings = embedding_model.encode(
    baseline_df["clean_conversation"].tolist(),
    convert_to_numpy=True
)

print("Baseline embedding shape:", baseline_embeddings.shape)

Baseline embedding shape: (5, 384)


In [16]:
new_conversation = """
Customer: I have been waiting for my reimbursement for a long time.
Agent: I understand how frustrating this delay must be. Let me check the status and help you with it.
"""

In [17]:
new_clean = preprocess_text(new_conversation)

print(new_clean)

i have been waiting for my reimbursement for a long time i understand how frustrating this delay must be let me check the status and help you with it


In [18]:
new_embedding = embedding_model.encode(
    [new_clean],
    convert_to_numpy=True
)

print(new_embedding.shape)

(1, 384)


In [19]:
similarities = cosine_similarity(
    new_embedding,
    baseline_embeddings
)

print(similarities)

[[0.6091391 0.5645975 0.4810368 0.6197611 0.521219 ]]


In [20]:
similarity_scores = similarities[0]

for session_id, score in zip(
    baseline_df["session_id"],
    similarity_scores
):
    print(session_id, round(score, 4))

S001 0.6091
S002 0.5646
S003 0.481
S004 0.6198
S005 0.5212


In [21]:
max_similarity = similarity_scores.max()

print("Maximum similarity:", round(max_similarity, 4))

Maximum similarity: 0.6198


In [22]:
positive_conversations = [
    "I understand your frustration with the delay. Let me check the status of your claim.",
    "I completely understand your concern. I will look into the issue for you.",
    "I can understand why this situation is worrying you. Let me see what I can do.",
    "I appreciate how frustrating this must be for you. I will check the claim immediately.",
    "I understand that you have been waiting for a long time. Let me investigate this.",
    "I can see why you are concerned about the delayed reimbursement. I will help you with this.",
    "I understand how upsetting this situation must be. Let me review your claim details.",
    "I hear your concern about the delay, and I will look into it right away.",
    "I understand why you are unhappy with the situation. Let me check this for you.",
    "I can appreciate how frustrating the delay has been. I will investigate the matter."
]

negative_conversations = [
    "Your claim is currently being processed.",
    "Please provide your policy number so I can check the claim.",
    "The standard processing time is ten business days.",
    "Your reimbursement request was submitted yesterday.",
    "I have checked the status and it is still under review.",
    "Your policy covers this type of claim.",
    "The claim department will contact you once the review is complete.",
    "Please upload the required documents to continue.",
    "Your claim number is CLM45892.",
    "The current status of your reimbursement is pending."
]

hard_negative_conversations = [
    "I understand that your claim has been pending for two weeks. The standard processing time is ten business days.",
    "I understand that you have been waiting for your reimbursement. Your request is still under review.",
    "I understand your concern regarding the claim. Unfortunately, the policy does not cover this situation.",
    "I understand that the delay has been frustrating, but there is no further action available at this stage.",
    "I understand your question about the claim. The status remains unchanged.",
    "I understand that you contacted us earlier. Please wait for the claims team to complete the review.",
    "I understand the issue you are referring to. Your policy documents are still being verified.",
    "I understand that you are asking about the delay. The claim is within the normal processing period.",
    "I understand your concern about the reimbursement. We are unable to provide an update at this time.",
    "I understand that you have been waiting. Please contact the claims department for further assistance."
]

In [23]:
positive_df = pd.DataFrame({
    "conversation": positive_conversations,
    "kpi_met": 1,
    "example_type": "positive"
})

negative_df = pd.DataFrame({
    "conversation": negative_conversations,
    "kpi_met": 0,
    "example_type": "negative"
})

hard_negative_df = pd.DataFrame({
    "conversation": hard_negative_conversations,
    "kpi_met": 0,
    "example_type": "hard_negative"
})

df_large = pd.concat(
    [positive_df, negative_df, hard_negative_df],
    ignore_index=True
)

df_large.insert(
    0,
    "session_id",
    [f"S{i:03d}" for i in range(1, len(df_large) + 1)]
)

df_large

,session_id,conversation,kpi_met,example_type
0,S001,I understand your frustration with the delay. Let me check the status of your claim.,1,positive
1,S002,I completely understand your concern. I will look into the issue for you.,1,positive
2,S003,I can understand why this situation is worrying you. Let me see what I can do.,1,positive
3,S004,I appreciate how frustrating this must be for you. I will check the claim immediately.,1,positive
4,S005,I understand that you have been waiting for a long time. Let me investigate this.,1,positive
5,S006,I can see why you are concerned about the delayed reimbursement. I will help you with this.,1,positive
6,S007,I understand how upsetting this situation must be. Let me review your claim details.,1,positive
7,S008,"I hear your concern about the delay, and I will look into it right away.",1,positive
8,S009,I understand why you are unhappy with the situation. Let me check this for you.,1,positive
9,S010,I can appreciate how frustrating the delay has been. I will investigate the matter.,1,positive


In [24]:
df_large["example_type"].value_counts()

,count
example_type,
positive,10
negative,10
hard_negative,10


In [25]:
df_large["clean_conversation"] = (
    df_large["conversation"]
    .apply(preprocess_text)
)

df_large.head()

,session_id,conversation,kpi_met,example_type,clean_conversation
0,S001,I understand your frustration with the delay. Let me check the status of your claim.,1,positive,i understand your frustration with the delay let me check the status of your claim
1,S002,I completely understand your concern. I will look into the issue for you.,1,positive,i completely understand your concern i will look into the issue for you
2,S003,I can understand why this situation is worrying you. Let me see what I can do.,1,positive,i can understand why this situation is worrying you let me see what i can do
3,S004,I appreciate how frustrating this must be for you. I will check the claim immediately.,1,positive,i appreciate how frustrating this must be for you i will check the claim immediately
4,S005,I understand that you have been waiting for a long time. Let me investigate this.,1,positive,i understand that you have been waiting for a long time let me investigate this


In [26]:
all_embeddings = embedding_model.encode(
    df_large["clean_conversation"].tolist(),
    convert_to_numpy=True
)

print("Embedding shape:", all_embeddings.shape)

Embedding shape: (30, 384)


In [27]:
baseline_df = df_large[
    df_large["example_type"] == "positive"
].copy()

baseline_embeddings = embedding_model.encode(
    baseline_df["clean_conversation"].tolist(),
    convert_to_numpy=True
)

print("Number of baselines:", len(baseline_df))
print("Baseline embedding shape:", baseline_embeddings.shape)

Number of baselines: 10
Baseline embedding shape: (10, 384)


In [28]:
def predict_kpi(conversation, threshold=0.60):

    # Preprocess
    clean_text = preprocess_text(conversation)

    # Create embedding
    query_embedding = embedding_model.encode(
        [clean_text],
        convert_to_numpy=True
    )

    # Compare against all baselines
    similarities = cosine_similarity(
        query_embedding,
        baseline_embeddings
    )[0]

    # Find strongest match
    max_similarity = similarities.max()

    # Find which baseline produced it
    best_match_index = similarities.argmax()
    best_match_session = baseline_df.iloc[best_match_index]["session_id"]

    # Apply threshold
    prediction = int(max_similarity >= threshold)

    return {
        "prediction": prediction,
        "max_similarity": max_similarity,
        "best_match": best_match_session
    }

In [29]:
test_a = "I understand how frustrating this delay must be. Let me check the status of your claim."

predict_kpi(test_a, threshold=0.60)

{'prediction': 1,
 'max_similarity': np.float32(0.94312525),
 'best_match': 'S001'}

In [30]:
test_b = "Your claim is currently being processed."

predict_kpi(test_b, threshold=0.60)

{'prediction': 1,
 'max_similarity': np.float32(0.63453937),
 'best_match': 'S004'}

In [31]:
test_c = "I understand that your claim has been pending for two weeks. The standard processing time is ten business days."

predict_kpi(test_c, threshold=0.60)

{'prediction': 1,
 'max_similarity': np.float32(0.7050961),
 'best_match': 'S001'}

In [32]:
def predict_all(df_input, threshold=0.60):

    predictions = []
    max_scores = []
    best_matches = []

    for conversation in df_input["clean_conversation"]:

        query_embedding = embedding_model.encode(
            [conversation],
            convert_to_numpy=True
        )

        similarities = cosine_similarity(
            query_embedding,
            baseline_embeddings
        )[0]

        max_similarity = similarities.max()
        best_match_index = similarities.argmax()

        predictions.append(
            int(max_similarity >= threshold)
        )

        max_scores.append(max_similarity)

        best_matches.append(
            baseline_df.iloc[best_match_index]["session_id"]
        )

    result = df_input.copy()
    result["max_similarity"] = max_scores
    result["best_match"] = best_matches
    result["prediction"] = predictions

    return result

In [33]:
results = predict_all(df_large, threshold=0.60)

In [34]:
results[
    [
        "session_id",
        "example_type",
        "kpi_met",
        "max_similarity",
        "prediction"
    ]
]

,session_id,example_type,kpi_met,max_similarity,prediction
0,S001,positive,1,1.000000,1
1,S002,positive,1,1.000000,1
2,S003,positive,1,1.000000,1
3,S004,positive,1,1.000000,1
4,S005,positive,1,1.000000,1
5,S006,positive,1,1.000000,1
6,S007,positive,1,1.000000,1
7,S008,positive,1,1.000000,1
8,S009,positive,1,1.000000,1
9,S010,positive,1,1.000000,1


In [35]:
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)

y_true = results["kpi_met"]
y_pred = results["prediction"]

print("Accuracy :", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred))
print("Recall   :", recall_score(y_true, y_pred))
print("F1       :", f1_score(y_true, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

Accuracy : 0.5666666666666667
Precision: 0.43478260869565216
Recall   : 1.0
F1       : 0.6060606060606061

Confusion Matrix:
[[ 7 13]
 [ 0 10]]


In [36]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

thresholds = np.arange(0.40, 0.91, 0.05)

threshold_results = []

for threshold in thresholds:

    y_pred = (results["max_similarity"] >= threshold).astype(int)

    threshold_results.append({
        "threshold": round(threshold, 2),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0)
    })

threshold_df = pd.DataFrame(threshold_results)

threshold_df

,threshold,accuracy,precision,recall,f1
0,0.40,0.400000,0.357143,1.0,0.526316
1,0.45,0.466667,0.384615,1.0,0.555556
2,0.50,0.500000,0.400000,1.0,0.571429
3,0.55,0.500000,0.400000,1.0,0.571429
4,0.60,0.566667,0.434783,1.0,0.606061
5,0.65,0.666667,0.500000,1.0,0.666667
6,0.70,0.766667,0.588235,1.0,0.740741
7,0.75,0.933333,0.833333,1.0,0.909091
8,0.80,1.000000,1.000000,1.0,1.000000
9,0.85,1.000000,1.000000,1.0,1.000000


In [37]:
import faiss

# Normalize embeddings so inner product = cosine similarity
baseline_embeddings_normalized = baseline_embeddings.copy()

faiss.normalize_L2(baseline_embeddings_normalized)

# Create FAISS index
dimension = baseline_embeddings_normalized.shape[1]
index = faiss.IndexFlatIP(dimension)

# Add baseline embeddings
index.add(baseline_embeddings_normalized)

print("Number of vectors in FAISS:", index.ntotal)
print("Embedding dimension:", dimension)

Number of vectors in FAISS: 10
Embedding dimension: 384


In [38]:
new_embedding_normalized = new_embedding.copy()
faiss.normalize_L2(new_embedding_normalized)

scores, indices = index.search(new_embedding_normalized, k=3)

print("Scores:", scores)
print("Indices:", indices)

Scores: [[0.83102256 0.69445336 0.63267994]]
Indices: [[5 0 9]]


In [39]:
for idx, score in zip(indices[0], scores[0]):
    print(
        baseline_df.iloc[idx]["session_id"],
        round(float(score), 4)
    )

S006 0.831
S001 0.6945
S010 0.6327


In [40]:
def predict_kpi_faiss(conversation, threshold=0.75):

    # 1. Preprocess
    clean_text = preprocess_text(conversation)

    # 2. Create embedding
    query_embedding = embedding_model.encode(
        [clean_text],
        convert_to_numpy=True
    )

    # 3. Normalize for cosine similarity
    faiss.normalize_L2(query_embedding)

    # 4. Search FAISS
    scores, indices = index.search(query_embedding, k=3)

    # 5. Best match
    best_score = float(scores[0][0])
    best_index = int(indices[0][0])
    best_session = baseline_df.iloc[best_index]["session_id"]

    # 6. Apply threshold
    prediction = int(best_score >= threshold)

    return {
        "prediction": prediction,
        "similarity": best_score,
        "best_match": best_session
    }

In [41]:
test_result = predict_kpi_faiss(new_conversation, threshold=0.75)

print(test_result)

{'prediction': 1, 'similarity': 0.8310225605964661, 'best_match': 'S006'}


In [42]:
def predict_all_faiss(df_input, threshold=0.75):

    predictions = []
    max_scores = []
    best_matches = []

    # Encode all conversations at once
    query_embeddings = embedding_model.encode(
        df_input["clean_conversation"].tolist(),
        convert_to_numpy=True
    )

    # Normalize
    faiss.normalize_L2(query_embeddings)

    # Retrieve best match for every conversation
    scores, indices = index.search(query_embeddings, k=1)

    for score, idx in zip(scores[:, 0], indices[:, 0]):

        predictions.append(int(score >= threshold))
        max_scores.append(float(score))
        best_matches.append(
            baseline_df.iloc[int(idx)]["session_id"]
        )

    result = df_input.copy()
    result["max_similarity"] = max_scores
    result["best_match"] = best_matches
    result["prediction"] = predictions

    return result

In [43]:
faiss_results = predict_all_faiss(df_large, threshold=0.75)

faiss_results[
    ["session_id", "example_type", "kpi_met",
     "max_similarity", "best_match", "prediction"]
]

,session_id,example_type,kpi_met,max_similarity,best_match,prediction
0,S001,positive,1,1.000000,S001,1
1,S002,positive,1,1.000000,S002,1
2,S003,positive,1,1.000000,S003,1
3,S004,positive,1,1.000000,S004,1
4,S005,positive,1,1.000000,S005,1
5,S006,positive,1,1.000000,S006,1
6,S007,positive,1,1.000000,S007,1
7,S008,positive,1,1.000000,S008,1
8,S009,positive,1,1.000000,S009,1
9,S010,positive,1,1.000000,S010,1


In [44]:
y_true = faiss_results["kpi_met"]
y_pred = faiss_results["prediction"]

print("Accuracy :", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred, zero_division=0))
print("Recall   :", recall_score(y_true, y_pred, zero_division=0))
print("F1       :", f1_score(y_true, y_pred, zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

Accuracy : 0.9333333333333333
Precision: 0.8333333333333334
Recall   : 1.0
F1       : 0.9090909090909091

Confusion Matrix:
[[18  2]
 [ 0 10]]


In [45]:
faiss_results[
    (faiss_results["kpi_met"] == 0) &
    (faiss_results["prediction"] == 1)
][
    ["session_id", "example_type", "conversation",
     "max_similarity", "best_match"]
]

,session_id,example_type,conversation,max_similarity,best_match
23,S024,hard_negative,"I understand that the delay has been frustrating, but there is no further action available at this stage.",0.788005,S010
27,S028,hard_negative,I understand that you are asking about the delay. The claim is within the normal processing period.,0.769809,S001


In [46]:
comparison_model = SentenceTransformer("all-mpnet-base-v2")

print("Model loaded successfully")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded successfully


In [47]:
comparison_embeddings = comparison_model.encode(
    df_large["clean_conversation"].tolist(),
    convert_to_numpy=True
)

print("Embedding shape:", comparison_embeddings.shape)

Embedding shape: (30, 768)


In [48]:
comparison_baseline_embeddings = comparison_model.encode(
    baseline_df["clean_conversation"].tolist(),
    convert_to_numpy=True
)

print("Baseline embedding shape:", comparison_baseline_embeddings.shape)

Baseline embedding shape: (10, 768)


In [49]:
from sklearn.metrics.pairwise import cosine_similarity

comparison_similarity = cosine_similarity(
    comparison_embeddings,
    comparison_baseline_embeddings
)

print("Similarity matrix shape:", comparison_similarity.shape)

Similarity matrix shape: (30, 10)


In [51]:
comparison_max_similarity = comparison_similarity.max(axis=1)

comparison_best_index = comparison_similarity.argmax(axis=1)

comparison_best_match = [
    baseline_df.iloc[idx]["session_id"]
    for idx in comparison_best_index
]

comparison_results = df_large.copy()

comparison_results["max_similarity"] = comparison_max_similarity
comparison_results["best_match"] = comparison_best_match

comparison_results[
    ["session_id", "example_type", "kpi_met",
     "max_similarity", "best_match"]
].head(10)

,session_id,example_type,kpi_met,max_similarity,best_match
0,S001,positive,1,1.0,S001
1,S002,positive,1,1.0,S002
2,S003,positive,1,1.0,S003
3,S004,positive,1,1.0,S004
4,S005,positive,1,1.0,S005
5,S006,positive,1,1.0,S006
6,S007,positive,1,1.0,S007
7,S008,positive,1,1.0,S008
8,S009,positive,1,1.0,S009
9,S010,positive,1,1.0,S010


In [52]:
comparison_results[
    comparison_results["example_type"] != "positive"
][
    ["session_id", "example_type", "kpi_met",
     "max_similarity", "best_match"]
]

,session_id,example_type,kpi_met,max_similarity,best_match
10,S011,negative,0,0.760442,S001
11,S012,negative,0,0.693458,S007
12,S013,negative,0,0.477632,S010
13,S014,negative,0,0.722550,S006
14,S015,negative,0,0.493476,S010
15,S016,negative,0,0.544985,S007
16,S017,negative,0,0.644339,S001
17,S018,negative,0,0.403152,S001
18,S019,negative,0,0.699300,S007
19,S020,negative,0,0.729015,S006


In [53]:
threshold = 0.75

comparison_results["prediction"] = (
    comparison_results["max_similarity"] >= threshold
).astype(int)

y_true_comparison = comparison_results["kpi_met"]
y_pred_comparison = comparison_results["prediction"]

print("Accuracy :", accuracy_score(y_true_comparison, y_pred_comparison))
print("Precision:", precision_score(y_true_comparison, y_pred_comparison, zero_division=0))
print("Recall   :", recall_score(y_true_comparison, y_pred_comparison, zero_division=0))
print("F1       :", f1_score(y_true_comparison, y_pred_comparison, zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true_comparison, y_pred_comparison))

Accuracy : 0.8333333333333334
Precision: 0.6666666666666666
Recall   : 1.0
F1       : 0.8

Confusion Matrix:
[[15  5]
 [ 0 10]]


In [54]:
thresholds = np.arange(0.40, 0.91, 0.05)

mpnet_threshold_results = []

for threshold in thresholds:

    y_pred = (
        comparison_results["max_similarity"] >= threshold
    ).astype(int)

    mpnet_threshold_results.append({
        "threshold": round(threshold, 2),
        "accuracy": accuracy_score(
            y_true_comparison, y_pred
        ),
        "precision": precision_score(
            y_true_comparison, y_pred, zero_division=0
        ),
        "recall": recall_score(
            y_true_comparison, y_pred, zero_division=0
        ),
        "f1": f1_score(
            y_true_comparison, y_pred, zero_division=0
        )
    })

mpnet_threshold_df = pd.DataFrame(mpnet_threshold_results)

mpnet_threshold_df

,threshold,accuracy,precision,recall,f1
0,0.40,0.333333,0.333333,1.0,0.500000
1,0.45,0.366667,0.344828,1.0,0.512821
2,0.50,0.433333,0.370370,1.0,0.540541
3,0.55,0.533333,0.416667,1.0,0.588235
4,0.60,0.533333,0.416667,1.0,0.588235
5,0.65,0.566667,0.434783,1.0,0.606061
6,0.70,0.666667,0.500000,1.0,0.666667
7,0.75,0.833333,0.666667,1.0,0.800000
8,0.80,0.933333,0.833333,1.0,0.909091
9,0.85,1.000000,1.000000,1.0,1.000000


In [55]:
positive_conversations = [
    "I understand your frustration with the delay. Let me check the status of your claim.",
    "I completely understand your concern. I will look into the issue for you.",
    "I can understand why this situation is worrying you. Let me see what I can do.",
    "I appreciate how frustrating this must be. I will check the claim immediately.",
    "I understand that you have been waiting for a long time. Let me investigate this.",
    "I can see why you are concerned about the delayed reimbursement. I will help you with this.",
    "I understand how upsetting this situation must be. Let me review your claim details.",
    "I hear your concern about the delay, and I will look into it right away.",
    "I understand why you are unhappy with the situation. Let me check this for you.",
    "I can appreciate how frustrating the delay has been. I will investigate the matter."
]

negative_conversations = [
    "Your claim is currently being processed.",
    "Please provide your policy number so I can check the claim.",
    "The standard processing time is ten business days.",
    "Your reimbursement request was submitted yesterday.",
    "I have checked the status and it is still under review.",
    "Your policy covers this type of claim.",
    "The claim department will contact you once the review is complete.",
    "Please upload the required documents to continue.",
    "Your claim number is CLM45892.",
    "The current status of your reimbursement is pending."
]

hard_negative_conversations = [
    "I understand your concern, but the status remains unchanged.",
    "I understand that you have been waiting. Unfortunately, there is nothing further I can do.",
    "I understand your frustration. The claim is still within the normal processing period.",
    "I understand the issue you are referring to. Your claim remains under review.",
    "I understand your concern about the reimbursement. We are unable to provide an update.",
    "I understand that you contacted us earlier. Please continue to wait for the claims team.",
    "I understand why you are asking. The policy documents are still being verified.",
    "I understand your concern regarding the delay. The claim is still pending.",
    "I understand that this is frustrating. There is no further update available.",
    "I understand your question about the claim. The status remains unchanged."
]

positive_df = pd.DataFrame({
    "conversation": positive_conversations,
    "kpi_met": 1,
    "example_type": "positive"
})

negative_df = pd.DataFrame({
    "conversation": negative_conversations,
    "kpi_met": 0,
    "example_type": "negative"
})

hard_negative_df = pd.DataFrame({
    "conversation": hard_negative_conversations,
    "kpi_met": 0,
    "example_type": "hard_negative"
})

df_large = pd.concat(
    [positive_df, negative_df, hard_negative_df],
    ignore_index=True
)

df_large.insert(
    0,
    "session_id",
    [f"S{i:03d}" for i in range(1, len(df_large) + 1)]
)

df_large["clean_conversation"] = df_large["conversation"].apply(
    preprocess_text
)

print(df_large["example_type"].value_counts())

example_type
positive         10
negative         10
hard_negative    10
Name: count, dtype: int64


In [56]:
# Recreate embeddings using the corrected dataset
all_embeddings = embedding_model.encode(
    df_large["clean_conversation"].tolist(),
    convert_to_numpy=True
)

# Positive examples become our baseline examples
baseline_df = df_large[
    df_large["example_type"] == "positive"
].copy()

baseline_embeddings = embedding_model.encode(
    baseline_df["clean_conversation"].tolist(),
    convert_to_numpy=True
)

# Build FAISS index
baseline_embeddings_normalized = baseline_embeddings.copy()
faiss.normalize_L2(baseline_embeddings_normalized)

dimension = baseline_embeddings_normalized.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(baseline_embeddings_normalized)

print("All embeddings:", all_embeddings.shape)
print("Baseline embeddings:", baseline_embeddings.shape)
print("FAISS vectors:", index.ntotal)

All embeddings: (30, 384)
Baseline embeddings: (10, 384)
FAISS vectors: 10
